In [ ]:
# imports

In [1]:
import sys
sys.path.append('../../../common_code')

In [2]:
import sqlite3
from paths import PATH_ROOT
from common_code.db_operations.verb_transactions.filter_verb_transaction_tables import *
from common_code.db_operations.verb_transactions.transactions_filtering import *

Transaktsioonide filtreerimine kasutades funktsiooni *filter_verb_transaction_tables*.

In [3]:
# ühenduse loomine verbimustrite andmebaasiga
con = sqlite3.connect("C:/Users/liivas/Documents/Töö/verbirektisoonid/vp_data3.db")
cur = con.cursor()

In [4]:
# transaktsioonide andmebaasi lisamine (v32)
cur.execute('ATTACH DATABASE "C:/Users/liivas/Documents/Töö/verbirektisoonid/v32_data.db" AS v32')

In [5]:
# uue andmebaasi lisamine transaktsioonide jaoks, kust on eemaldatud olemasolevad verbimustrid
cur.execute('ATTACH DATABASE "v32_data_filtered2.db" AS filtered')

In [6]:
cur.execute('ATTACH DATABASE "C:/Users/liivas/Documents/Töö/verbirektisoonid/db_operations/verb_negations/negations.db" AS neg')

In [7]:
# uue andmebaasi lisamine filtreeritud transaktsioonide jaoks, kust on omakorda eemaldatud eitused
cur.execute('ATTACH DATABASE "v32_data_filtered_neg2.db" AS filtered_neg')

In [8]:
# uue andmebaasi lisamine filtreeritud transaktsioonide jaoks, kust on omakorda eemaldatud eitused
cur.execute('ATTACH DATABASE "v32_data_filtered_no_neg2.db" AS filtered_no_neg')

### Filtreeritud transaktsioonid

In [9]:
# transaktsioonide head ID-de tabeli loomine, mis vastavad transaktsioonidele, mis ei vasta olemasolevatele mustritele

create_filtered_head_id_tbl(cur,
                           all_ids_tbl='verb_matches', # andmebaasist vp_data3 mustritele vastavad verbid
                           head_id_col1='head_id',
                           ids_to_filter_tbl='verb_phrase_matches', # andmebaasist vp_data3 tervikmustreid sisaldavad fraasid
                           head_id_col2='head_id',
                           output_tbl='filtered.filtered_head_ids')

In [9]:
# filtreeritud transaktsioonide tabelite loomine
filter_verb_transaction_tables(
    conn=con,
    source_schema='v32',
    transaction_head='transaction_head',
    transaction_row='transaction_row',
    target_schema='filtered',
    new_transaction_head='transaction_head',
    new_transaction_row='transaction_row',
    ids_schema='filtered',
    ids_table='filtered_head_ids',
    ids_column='head_id',
    delete_if_exists= True,
    copy_indexes=True,
    verbose= True,
)

CREATE TABLE "filtered"."transaction_head" (`id` INTEGER PRIMARY KEY AUTOINCREMENT, `sentence_id` int, `loc` int, `verb` text, `verb_compound` text, `form` text, `deprel` text, `feats` text)
Created table 'filtered.transaction_head' (foreign keys ignored).
Copying indexes from 'v32.transaction_head' to 'filtered.transaction_head'
CREATE INDEX "filtered"."22ffa9c4_`transaction_head_deprel`" ON "transaction_head"("`deprel`" ASC)
Index '22ffa9c4_`transaction_head_deprel`' created successfully.
CREATE INDEX "filtered"."22ffa9c4_`transaction_head_feats`" ON "transaction_head"("`feats`" ASC)
Index '22ffa9c4_`transaction_head_feats`' created successfully.
CREATE INDEX "filtered"."22ffa9c4_`transaction_head_verb_compound`" ON "transaction_head"("`verb_compound`" ASC)
Index '22ffa9c4_`transaction_head_verb_compound`' created successfully.
CREATE INDEX "filtered"."22ffa9c4_`transaction_head_verb`" ON "transaction_head"("`verb`" ASC)
Index '22ffa9c4_`transaction_head_verb`' created successfully.


True

In [10]:
# transaction_row täiendav 'deprel' tulba väärtuste filtreerimine

remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='nsubj')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='advmod')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='advcl')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='csubj')

In [11]:
# transaction_row hulgast abiverbidest AUX-de eemaldamine

remove_aux_verbs(cur,
                 transaction_row='filtered.transaction_row')

### Filtreeritud transaktsioonid ainult eitustega

In [12]:
# filtreeritud transaktsioonide tabelite loomine
filter_verb_transaction_tables(
    conn=con,
    source_schema='filtered',
    transaction_head='transaction_head',
    transaction_row='transaction_row',
    target_schema='filtered_neg',
    new_transaction_head='transaction_head',
    new_transaction_row='transaction_row',
    ids_schema='neg',
    ids_table='neg_phrase_matches',
    ids_column='head_id',
    delete_if_exists= True,
    copy_indexes=True,
    verbose= True,
)

CREATE TABLE "filtered_neg"."transaction_head" (`id` INTEGER PRIMARY KEY AUTOINCREMENT, `sentence_id` int, `loc` int, `verb` text, `verb_compound` text, `form` text, `deprel` text, `feats` text)
Created table 'filtered_neg.transaction_head' (foreign keys ignored).
Copying indexes from 'filtered.transaction_head' to 'filtered_neg.transaction_head'
CREATE UNIQUE INDEX "filtered_neg"."6482d49b_22ffa9c4_transaction_head_uniq" ON "transaction_head"(sentence_id, loc)
Index '6482d49b_22ffa9c4_transaction_head_uniq' created successfully.
CREATE INDEX "filtered_neg"."6482d49b_22ffa9c4_`transaction_head_verb`" ON "transaction_head"("`verb`" ASC)
Index '6482d49b_22ffa9c4_`transaction_head_verb`' created successfully.
CREATE INDEX "filtered_neg"."6482d49b_22ffa9c4_`transaction_head_verb_compound`" ON "transaction_head"("`verb_compound`" ASC)
Index '6482d49b_22ffa9c4_`transaction_head_verb_compound`' created successfully.
CREATE INDEX "filtered_neg"."6482d49b_22ffa9c4_`transaction_head_feats`" ON "

True

### Filtreeritud transaktsioonid ilma eitusteta

In [13]:
# transaktsioonide ID-de tabeli loomine, mis vastavad transaktsioonidele, mis pole eituste hulgas

create_filtered_head_id_tbl(cur,
                           'filtered.transaction_head', # esimesest filtreeritud andmebaasist head ID-d
                           'id',
                           'filtered_neg.transaction_head', # ainult eitusi sisaldavast filtreeritud andmebaasist head ID-d
                           'id',
                           'filtered_no_neg.filtered_head_ids')

In [14]:
# filtreeritud transaktsioonide tabelite loomine
filter_verb_transaction_tables(
    conn=con,
    source_schema='filtered',
    transaction_head='transaction_head',
    transaction_row='transaction_row',
    target_schema='filtered_no_neg',
    new_transaction_head='transaction_head',
    new_transaction_row='transaction_row',
    ids_schema='filtered_no_neg',
    ids_table='filtered_head_ids',
    ids_column='head_id',
    delete_if_exists= True,
    copy_indexes=True,
    verbose= True,
)

CREATE TABLE "filtered_no_neg"."transaction_head" (`id` INTEGER PRIMARY KEY AUTOINCREMENT, `sentence_id` int, `loc` int, `verb` text, `verb_compound` text, `form` text, `deprel` text, `feats` text)
Created table 'filtered_no_neg.transaction_head' (foreign keys ignored).
Copying indexes from 'filtered.transaction_head' to 'filtered_no_neg.transaction_head'
CREATE UNIQUE INDEX "filtered_no_neg"."037f72a6_22ffa9c4_transaction_head_uniq" ON "transaction_head"(sentence_id, loc)
Index '037f72a6_22ffa9c4_transaction_head_uniq' created successfully.
CREATE INDEX "filtered_no_neg"."037f72a6_22ffa9c4_`transaction_head_verb`" ON "transaction_head"("`verb`" ASC)
Index '037f72a6_22ffa9c4_`transaction_head_verb`' created successfully.
CREATE INDEX "filtered_no_neg"."037f72a6_22ffa9c4_`transaction_head_verb_compound`" ON "transaction_head"("`verb_compound`" ASC)
Index '037f72a6_22ffa9c4_`transaction_head_verb_compound`' created successfully.
CREATE INDEX "filtered_no_neg"."037f72a6_22ffa9c4_`transact

True

In [15]:
con.close()

NB! Andmebaasi *v32_data_filtered_no_neg2.db* tabelitesse *transaction_head* ja *transaction_row* on mõned eitust sisaldavad transaktsioonid alles jäänud. Täpne põhjus on selgumisel, aga näib, et paljudel juhtudel puudub *transaction_head* tabelis olevatel neg verbidel fraasisisu *transaction_row* tabelis (ehk need verbid on üksi, võimalik, et varasema deprelite filtreerimise tagajärjel).